# PAV_Preprocessing

Pavlovian preprocessing: raw photometry and video tracking → one merged CSV per session.

**Pipeline:** **1. PAV_Preprocessing** → 2. PAV_Analysis → 3. PAV_Publication_Figure

### Inputs (read from this repo)
| Path | Contents |
|---|---|
| `Photometry/Pavlovian/RWD/RWD_Fluorescence/` | RWD photometry exports (410 / 470 nm, TTL events), 48 sessions |
| `Photometry/Pavlovian/Bonsai/Scaled_Renamed/` | Cleaned Bonsai tracking with trial flags, 44 sessions |

### Outputs (downloaded as a zip at the end)
- One merged CSV per session (`…_merged.csv`, or `…_merged_NOBONSAIDATA.csv` when there is no video), for `Photometry/Pavlovian/RWD/Processed Data/`
- `preprocessing_qc.csv` and a QC figure per session

### How to run
Run all cells top to bottom. `SKIP_EXISTING=True` skips sessions already in `Processed Data`; set it to `False` to rebuild every session from raw. `VERIFY_VS_EXISTING=True` compares each rebuilt file with the committed one. The run is pinned to the current commit of `main`, which is printed.

### Stages
- **A** Load cleaned Bonsai tracking; collapse duplicated frames and recompute speed where needed
- **B** Photometry: double-exponential debleaching → motion correction against 410 nm → z-score relative to the lowest 25% of samples (`Corrected_Fluorescence_Z_25`)
- **C** Events: `ToneStart` / `PelletGrab` from the TTL record (pulse < 300 ms = tone, ≥ 300 ms = grab)
- **D** Align the Bonsai clock to the photometry clock and interpolate tracking onto it
- **E** Position: locate the feeder from pellet-grab positions and convert to cm (`DistFromFED`)
- **F** Run all sessions, QC, verify, save

Four sessions have photometry but no video and are processed without tracking.

In [ ]:
# @title Setup — imports
import io, os, re, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.optimize import curve_fit
from scipy.stats import linregress
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm
print("Imports OK.")


Imports OK.


In [ ]:
# @title Output paths & run options
# Inputs are read from the GitHub repo; outputs are written to the Colab runtime
# and downloaded at the end. Commit the outputs to the repo yourself.

OUT_DIR = '/content/PAV_Preprocessing_out'  # Colab-local scratch, zipped at the end
QC_DIR  = f'{OUT_DIR}/QC'                   # per-animal QC figures
os.makedirs(QC_DIR, exist_ok=True)

# Skip sessions whose merged CSV is already committed to GitHub ('Processed Data').
# Only 24 of the 48 sessions were ever processed (all session _02), so with
# SKIP_EXISTING=True this notebook processes the 24 that are missing.
# Set False to rebuild every session from raw and verify against the committed ones.
SKIP_EXISTING = False

# Compare each freshly processed animal against the ORIGINAL merged data, pinned
# so a run is pinned to one version of the data and prints which one it used.
VERIFY_VS_EXISTING = True

print(f"Output -> {OUT_DIR}\nQC     -> {QC_DIR}\n"
      f"SKIP_EXISTING={SKIP_EXISTING}  VERIFY_VS_EXISTING={VERIFY_VS_EXISTING}")


Output -> /content/PAV_Preprocessing_out
QC     -> /content/PAV_Preprocessing_out/QC
SKIP_EXISTING=False  VERIFY_VS_EXISTING=True


In [ ]:
# @title Parameters & configuration
GITHUB_OWNER = "CreedLab-Neuro"   # or "MasBarr"
GITHUB_REPO  = "Manuscript_PMID41279070"
GH_API  = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
GH_BASE = f"{GH_API}/contents"
PAV     = "Photometry/Pavlovian"

# ── Pin this run to one commit ────────────────────────────────────────────────
# raw.githubusercontent caches BRANCH urls (~5 min) and different edge nodes
# expire independently, so fetching ".../main/..." right after a data push can
# hand you a mix of old and new files -- silently, in the same run. Resolving
# main to its commit SHA once and fetching ".../<sha>/..." avoids this entirely:
# SHA urls are immutable, so a cache hit is always correct.
# It also makes the run self-documenting -- the printed SHA is exactly which
# version of the data produced these outputs.
DATA_SHA = requests.get(f"{GH_API}/commits/main").json()['sha']

API_SCALED    = f"{GH_BASE}/{PAV}/Bonsai/Scaled_Renamed?ref={DATA_SHA}"
API_RAW       = f"{GH_BASE}/{PAV}/Bonsai/Raw?ref={DATA_SHA}"
API_RWD_FLUOR = f"{GH_BASE}/{PAV}/RWD/RWD_Fluorescence?ref={DATA_SHA}"
# NB: the committed output folder has a SPACE in its name.
API_MERGED    = f"{GH_BASE}/{PAV}/RWD/Processed%20Data?ref={DATA_SHA}"

# VERIFY_VS_EXISTING compares each rebuilt session with the committed 'Processed Data'.
API_MERGED_REF = API_MERGED

# Bonsai comes from Scaled_Renamed: unlike Variable Tone we do NOT re-clean from raw,
# because those files carry the hand-curated Completed_Tone / Completed_PelletGrab trial
# exclusions, which cannot be regenerated from the raw tracking.
BONSAI_SOURCE = 'scaled'

# Group and genotype are in the filename, so parse them rather than hardcoding a map:
#   Fluorescence_F18_SHAM_DAT_091525_01.csv
#   OFP_Photo_Bonsai_F18_SHAM_DAT_091525_01_scaled.csv
_NAME_RE = re.compile(r'([MF]\d+)_(SHAM|SNI|SK3)_(C57|DAT)_(\d{6})_(\d{2})')

def parse_name(fname):
    """-> dict(animal, group, genotype, date, session) or None."""
    m = _NAME_RE.search(fname)
    if not m: return None
    return {'animal':m.group(1), 'group':m.group(2), 'genotype':m.group(3),
            'date':m.group(4), 'session':m.group(5)}

def session_key(fname):
    """Pairing key for RWD <-> Bonsai <-> merged.

    Deliberately animal+session and NOT the date: session 02 for F2 (073025) and
    M2 (071325) is filed under a 070225 stamp, so the RWD and Bonsai dates still
    disagree there. Stage F reports any mismatch rather than silently trusting it."""
    d = parse_name(fname)
    return f"{d['animal']}_{d['session']}" if d else None

# Pavlovian encodes only TWO events, split on pulse width (see RWD_Bonsai_merge):
#   < 300 ms = ToneStart,  >= 300 ms = PelletGrab
# Every Pavlovian session used 5 s tones, so there is no duration split.
THRESHOLD_MS = 300
TONE_DUR_S   = 5.0
FED_X_CM     = 9.0   # starting anchor only; Stage E relocates the FED

# Stage E thresholds
NEAR_FED_CM=8.0; SPEED_MAX=80.0; MIN_REG_PTS=100
R2_FLAG=0.70; MAE_FLAG=2.0; ANCHOR_COL='DistFromFED_orig'; OVERWRITE_DISTFROMFED=True

# Stage D column roles:
# - BONSAI_ALIGN: Bonsai events used to align the clock (proximity-matched to RWD; incl. tones)
# - BONSAI_SNAP : Bonsai binary columns written to the merged CSV (tones now come from RWD)
# - BONSAI_CONT : continuous tracking interpolated onto the RWD axis
BONSAI_ALIGN =['ToneStart_Bonsai','PelletGrab_Bonsai']
# Completed_* are the hand-curated trial exclusions baked into Scaled_Renamed. They are
# carried through unchanged -- they encode decisions that cannot be recomputed.
BONSAI_SNAP  =['ToneStart_Bonsai','PelletGrab_Bonsai','Completed_Tone','Completed_PelletGrab']
BONSAI_CONT  =['Xpos','Ypos','Speed_cm','Speed_px','DistFromFED']
RWD_EVENT_COLS=['ToneStart','PelletGrab']

def get_animal_id(s):
    m=re.search(r'([MF]\d+)', s); return m.group(1) if m else None
def get_animal_num(aid):
    m=re.search(r'\d+', aid or ''); return int(m.group()) if m else 999
print(f"Config ready. BONSAI_SOURCE = {BONSAI_SOURCE}")
print(f"Data version (main @ commit): {DATA_SHA[:10]}")


In [ ]:
# @title GitHub loaders
def gh_list(api_url):
    r=requests.get(api_url); r.raise_for_status()
    return [{'name':f['name'],'download_url':f['download_url']}
            for f in r.json() if f['name'].endswith('.csv')]

def gh_read(url, **kw):
    r=requests.get(url); r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), low_memory=False, **kw)

def gh_load_folder_keyed(api_url, keyfn, desc="Loading"):
    """Like gh_load_folder but keyed by an arbitrary function of the filename.

    Pavlovian has two sessions per animal, so an animal-ID key would collide and
    silently keep only one of them."""
    files=gh_list(api_url); out={}
    def _one(f): return keyfn(f['name']), f['name'], gh_read(f['download_url'])
    with ThreadPoolExecutor(max_workers=10) as ex:
        futs={ex.submit(_one,f):f['name'] for f in files}
        with tqdm(total=len(files), desc=desc) as pbar:
            for fut in as_completed(futs):
                try:
                    k,name,df=fut.result()
                    if k:
                        if k in out: print(f"  ! duplicate key {k}: {name} overwrites {out[k]['name']}")
                        out[k]={'name':name,'df':df}
                except Exception as e: print(f"  x {futs[fut]}: {e}")
                pbar.update(1)
    return out

def gh_load_folder(api_url, desc="Loading"):
    files=gh_list(api_url); out={}
    def _one(f): return get_animal_id(f['name']), f['name'], gh_read(f['download_url'])
    with ThreadPoolExecutor(max_workers=10) as ex:
        futs={ex.submit(_one,f):f['name'] for f in files}
        with tqdm(total=len(files), desc=desc) as pbar:
            for fut in as_completed(futs):
                try:
                    aid,name,df=fut.result()
                    if aid: out[aid]={'name':name,'df':df}
                except Exception as e: print(f"  x {futs[fut]}: {e}")
                pbar.update(1)
    return out
print("Loaders defined.")


Loaders defined.


In [ ]:
# @title Stage A — load cleaned Bonsai (Scaled_Renamed) + CombineLatest de-dup
# Scaled_Renamed already carries Speed_px, Speed_cm, DistFromFED and the hand-curated
# Completed_Tone / Completed_PelletGrab flags, so those are normally carried through.
#
# BUT: M26 / M27 / M28 were recorded with the Bonsai workflow where each observable
# fires independently, so every frame appears 4x under one timestamp (all 6 of their
# sessions). Their stored Speed_cm is unusable -- every "finite" value is exactly 0.0,
# with ~212k infs and ~107k NaNs per file, because dt = 0 on 3 of every 4 rows.
# Variable Tone fixed this inside clean_bonsai_raw; here the damage is already baked
# into the cleaned file, so we de-duplicate AND recompute the speed columns.
# Guarded, so it is a strict no-op for every session that is already 1 row/frame.

_BIN_COLS = ['ToneStart_Bonsai', 'PelletGrab_Bonsai', 'Completed_Tone', 'Completed_PelletGrab']

def dedup_bonsai(df, tag=''):
    """Collapse 4x-duplicated frames and rebuild the speed columns."""
    if 'Timestamp_Bonsai' not in df.columns or not df['Timestamp_Bonsai'].duplicated().any():
        return df, False
    n0 = len(df)
    d = df.sort_values('Timestamp_Bonsai', kind='stable').copy()
    # OR the binary flags across each timestamp group so no event sample is lost
    for c in _BIN_COLS:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors='coerce').fillna(0)
            d[c] = d.groupby('Timestamp_Bonsai')[c].transform('max')
    d = d.drop_duplicates('Timestamp_Bonsai', keep='last').reset_index(drop=True)

    # the stored speeds cannot be salvaged (they were divided by dt = 0), so rebuild
    dt = d['Timestamp_Bonsai'].diff()
    if {'Xscaled', 'Yscaled'}.issubset(d.columns):
        d['Speed_cm'] = np.sqrt(d['Xscaled'].diff()**2 + d['Yscaled'].diff()**2) / dt
        d.loc[d.index[0], 'Speed_cm'] = 0.0
    if {'Xpos', 'Ypos'}.issubset(d.columns) and 'Speed_px' in d.columns:
        d['Speed_px'] = np.sqrt(d['Xpos'].diff()**2 + d['Ypos'].diff()**2) / dt
        d.loc[d.index[0], 'Speed_px'] = 0.0
    _med = np.nanmedian(d['Speed_cm'].values[np.isfinite(d['Speed_cm'].values)])            if 'Speed_cm' in d.columns else float('nan')
    print(f"    {tag}: CombineLatest de-dup {n0} -> {len(d)} rows, "
          f"Speed_cm rebuilt (median {_med:.2f} cm/s)")
    return d, True

_b = gh_load_folder_keyed(API_SCALED, session_key, desc="Stage A: cleaned Bonsai")
bonsai_by_session, bonsai_names, _fixed = {}, {}, []
for k, v in _b.items():
    df, was = dedup_bonsai(v['df'], k)
    bonsai_by_session[k] = df
    bonsai_names[k] = v['name']
    if was: _fixed.append(k)

print(f"Loaded cleaned Bonsai for {len(bonsai_by_session)} sessions.")
print(f"  de-duplicated: {len(_fixed)} {sorted(_fixed)}")
_have = sum(1 for d in bonsai_by_session.values() if 'Completed_Tone' in d.columns)
print(f"  carrying Completed_Tone / Completed_PelletGrab: {_have}")


Stage A: cleaned Bonsai:   0%|          | 0/44 [00:00<?, ?it/s]

    M12_01: CombineLatest de-dup 316135 -> 79024 rows, Speed_cm rebuilt (median 1.47 cm/s)
    M11_01: CombineLatest de-dup 370741 -> 92682 rows, Speed_cm rebuilt (median 2.22 cm/s)
    M13_01: CombineLatest de-dup 377591 -> 94394 rows, Speed_cm rebuilt (median 1.07 cm/s)
    M26_01: CombineLatest de-dup 425755 -> 106436 rows, Speed_cm rebuilt (median 1.77 cm/s)
    M27_01: CombineLatest de-dup 426157 -> 106536 rows, Speed_cm rebuilt (median 2.77 cm/s)
    M27_02: CombineLatest de-dup 435737 -> 108919 rows, Speed_cm rebuilt (median 2.45 cm/s)
    M26_02: CombineLatest de-dup 488287 -> 122056 rows, Speed_cm rebuilt (median 2.52 cm/s)
    M7_01: CombineLatest de-dup 320429 -> 80101 rows, Speed_cm rebuilt (median 2.31 cm/s)
    M28_02: CombineLatest de-dup 444859 -> 111196 rows, Speed_cm rebuilt (median 2.01 cm/s)
    M28_01: CombineLatest de-dup 517083 -> 129269 rows, Speed_cm rebuilt (median 2.60 cm/s)
    M8_01: CombineLatest de-dup 316371 -> 79090 rows, Speed_cm rebuilt (median 1.44 c

In [ ]:
# @title Stage B — photometry functions (debleach -> motion -> 25th-pctile z)
def double_exponential(t, const, amp_fast, amp_slow, tau_slow, tau_multiplier):
    tau_fast=tau_slow*tau_multiplier
    return const+amp_slow*np.exp(-t/tau_slow)+amp_fast*np.exp(-t/tau_fast)

def load_rwd_raw(download_url):
    txt=requests.get(download_url); txt.raise_for_status(); txt=txt.text
    for h in (0,1):
        d=pd.read_csv(io.StringIO(txt), header=h, low_memory=False)
        if ('TimeStamp' in d.columns or 'Time' in d.columns) and \
           ('Events' in d.columns or 'CH1-470' in d.columns or 'Fluorescence' in d.columns):
            return d
    return pd.read_csv(io.StringIO(txt), low_memory=False)

def process_photometry(df_raw):
    df=df_raw.copy()
    if 'Time' in df.columns and 'TimeStamp' not in df.columns: df=df.rename(columns={'Time':'TimeStamp'})
    if 'CH1-410' in df.columns: df=df.rename(columns={'CH1-410':'UV'})
    if 'CH1-470' in df.columns: df=df.rename(columns={'CH1-470':'Fluorescence'})
    df=df.loc[:, ~df.columns.str.contains('^Unnamed', regex=True)]
    df['TimeStampAdjusted']=df['TimeStamp']/1000 if df['TimeStamp'].max()>60000 else df['TimeStamp']
    t=df['TimeStampAdjusted']
    def _fit(sig):
        mx=np.max(sig); bounds=([0,0,0,0,0],[mx,mx,mx,36000,1]); p0=[mx/2,mx/4,mx/4,7500,0.1]
        p,_=curve_fit(double_exponential,t,sig,p0=p0,bounds=bounds,maxfev=1000)
        return double_exponential(t,*p)
    fl,uv=df['Fluorescence'],df['UV']
    fl_db=fl-_fit(fl); uv_db=uv-_fit(uv)
    slope,intercept,*_=linregress(x=uv_db,y=fl_db)
    est=intercept+slope*uv_db; corr=fl_db-est
    p25=np.percentile(corr,25); bottom=corr[corr<=p25]
    z25=(corr-bottom.mean())/bottom.std()
    df['Debleached Fluorescence']=fl_db; df['Debleached UV']=uv_db
    df['Estimated Motion']=est; df['CorrectedFluorescence']=corr
    df['Corrected_Fluorescence_Z_25']=z25
    df['CorrectedFluorescence_Z']=(corr-corr.mean())/corr.std()
    return df
print("Stage B functions defined.")


Stage B functions defined.


In [ ]:
# @title Stage C — RWD events (ToneStart / PelletGrab)
# Most raw Pavlovian files already carry ToneStart and PelletGrab. Two do not
# (F2 and M2, session 070225_02) and are parsed from the Events column here.
#
# Encoding, from the original RWD_Bonsai_merge notebook: each event is a pulse
# delimited by "Input1*2*0" (rise) and "Input1*2*1" (fall). Pulse width splits the
# two event types at THRESHOLD_MS -- under 300 ms is a tone, 300 ms or more is a
# pellet grab. A rise and fall landing on the SAME sample is a very short pulse,
# i.e. a tone.

def derive_rwd_events(df):
    """Parse ToneStart / PelletGrab out of the Events column."""
    df['ToneStart']  = 0
    df['PelletGrab'] = 0
    ev    = df['Events'].fillna('').astype(str)
    idx_0 = df.index[ev.str.contains(r'Input1\*2\*0', regex=True)].tolist()
    idx_1 = df.index[ev.str.contains(r'Input1\*2\*1', regex=True)].tolist()
    j = 0
    for i in idx_0:
        if 'Input1*2*1' in ev.at[i]:          # rise and fall on one sample -> tone
            df.at[i,'ToneStart'] = 1; continue
        while j < len(idx_1) and idx_1[j] <= i:
            j += 1
        if j >= len(idx_1):                   # no fall recorded -> treat as tone
            df.at[i,'ToneStart'] = 1; continue
        width = df.at[idx_1[j],'TimeStamp'] - df.at[i,'TimeStamp']
        df.at[i, 'ToneStart' if width < THRESHOLD_MS else 'PelletGrab'] = 1
    return df

def add_rwd_events(df):
    """Carry through the committed event columns, or derive them if absent."""
    have = 'ToneStart' in df.columns and 'PelletGrab' in df.columns
    if have:
        df['ToneStart']  = pd.to_numeric(df['ToneStart'],  errors='coerce').fillna(0).astype(int)
        df['PelletGrab'] = pd.to_numeric(df['PelletGrab'], errors='coerce').fillna(0).astype(int)
        df.attrs['events_source'] = 'carried'
    elif 'Events' in df.columns:
        df = derive_rwd_events(df)
        df.attrs['events_source'] = 'derived'
    else:
        df['ToneStart'] = 0; df['PelletGrab'] = 0
        df.attrs['events_source'] = 'NONE'
    return df
print("Stage C helpers defined.")


Stage C helpers defined.


In [ ]:
# @title Stage D — align Bonsai->RWD clock + interpolate (functions)
def _event_times(df, cols, tcol):
    cols=[c for c in cols if c in df.columns]
    if not cols: return np.array([])
    return np.sort(df.loc[(df[cols].sum(axis=1)>0).values, tcol].values)

def align_and_merge(photo_df, bonsai_df):
    """Align the Bonsai clock to RWD by PROXIMITY-matching shared events (robust to a
    Bonsai recording that dropped events), then interpolate tracking + snap the Bonsai
    dispense/retrieval markers onto the RWD axis. Tone markers come from RWD (Stage C)."""
    df=photo_df.copy()
    rwd_ev=_event_times(photo_df, RWD_EVENT_COLS, 'TimeStampAdjusted')
    bon_ev=_event_times(bonsai_df, BONSAI_ALIGN, 'Timestamp_Bonsai')
    if len(rwd_ev)<2 or len(bon_ev)<2: return df, None

    # ── Coarse offset: anchor a SPREAD of Bonsai events (not just the first few —
    # some recordings have spurious extra events at the start) against every RWD
    # event. Covers any offset (some Bonsai clocks are shifted by >1000 s) and is
    # robust to junk events anywhere. Pick the shift matching the most events. ──
    rr=rwd_ev[None,:]
    best_off, best_n = 0.0, -1
    _aidx=np.unique(np.linspace(0, len(bon_ev)-1, min(10, len(bon_ev))).astype(int))
    for ba in bon_ev[_aidx]:
        for rt in rwd_ev:
            off=rt-ba
            d=np.min(np.abs((bon_ev[:,None]+off)-rr), axis=1)
            n=int((d<0.5).sum())
            if n>best_n: best_n, best_off = n, off

    # ── Proximity match + linear (offset+drift) fit on matched pairs only ────────
    shifted=bon_ev+best_off
    j=np.argmin(np.abs(shifted[:,None]-rr), axis=1)
    keep=np.abs(shifted-rwd_ev[j])<1.0
    mb, mr = bon_ev[keep], rwd_ev[j[keep]]
    if len(mb)<3: return df, None
    slope,intercept,r,_,_=linregress(mb, mr)

    # ── Refine: re-match using the fitted line (accounts for drift) and refit ────
    for _ in range(5):
        pred=slope*bon_ev+intercept
        j2=np.argmin(np.abs(pred[:,None]-rr), axis=1)
        keep2=np.abs(pred-rwd_ev[j2])<1.0
        if int(keep2.sum())<3: break
        mb2, mr2 = bon_ev[keep2], rwd_ev[j2[keep2]]
        s2,i2,r2,_,_=linregress(mb2, mr2)
        converged = abs(s2-slope)<1e-9 and abs(i2-intercept)<1e-6
        mb, mr, slope, intercept, r = mb2, mr2, s2, i2, r2
        if converged: break

    # Too few matched events to trust a slope -> offset-only (assume no drift).
    if len(mb) < 8:
        slope = 1.0; intercept = float(np.median(mr - mb)); r = float('nan')

    bcorr=bonsai_df['Timestamp_Bonsai'].values*slope+intercept
    rwd_t=df['TimeStampAdjusted'].values
    for c in BONSAI_CONT:
        if c in bonsai_df.columns:
            f=interp1d(bcorr, bonsai_df[c].values, kind='linear', bounds_error=False, fill_value=np.nan)
            df[c]=f(rwd_t)
    for c in BONSAI_SNAP:
        if c in bonsai_df.columns:
            arr=np.zeros(len(df),dtype=int)
            for t in bcorr[bonsai_df[c].values==1]: arr[np.argmin(np.abs(rwd_t-t))]=1
            df[c]=arr
    info={'offset':intercept,'slope':slope,'r':r,'n_match':int(len(mb)),
          'n_rwd_ev':len(rwd_ev),'n_bon_ev':len(bon_ev),'drift_s_per_hr':(1-slope)*3600,
          'bcorr':bcorr,'rwd_ev':rwd_ev,'bon_ev':bon_ev,'mb':mb,'mr':mr,'best_off':best_off}
    return df, info
print("Stage D functions defined.")


Stage D functions defined.


In [ ]:
# @title Stage E — position normalization (function)
def normalize_positions(df, animal_id, group='?', session=''):
    # FED position is estimated from where the animal actually collects pellets
    # (PelletGrab), replacing the old hardcoded x = 9 cm assumption.
    df=df.copy()
    rec={'Animal':animal_id,'Group':group,'Session':session,'Status':'NO_BONSAI','FED_x_px':np.nan,
         'FED_y_px':np.nan,'scale_cm_per_px':np.nan,'R2':np.nan,'MAE_cm':np.nan,'n_grab':0}
    if 'Xpos' not in df.columns or 'Ypos' not in df.columns: return df, rec
    x_raw=pd.to_numeric(df['Xpos'],errors='coerce').values
    y_raw=pd.to_numeric(df['Ypos'],errors='coerce').values
    anchor=ANCHOR_COL if ANCHOR_COL in df.columns else 'DistFromFED'
    dist=pd.to_numeric(df[anchor],errors='coerce').values
    valid_xy=~(np.isnan(x_raw)|np.isnan(y_raw))
    if 'PelletGrab_Bonsai' not in df.columns: rec['Status']='NO_GRAB'; return df, rec
    retr=(pd.to_numeric(df['PelletGrab_Bonsai'],errors='coerce')==1).values & valid_xy
    rec['n_grab']=int(retr.sum())
    dr=dist[retr]; near=retr.copy(); near[retr]=(dr<NEAR_FED_CM)|np.isnan(dr)
    if near.sum()<3: near=retr
    fed_x=float(np.nanmedian(x_raw[near])); fed_y=float(np.nanmedian(y_raw[near]))
    rec['FED_x_px']=round(fed_x,1); rec['FED_y_px']=round(fed_y,1)
    xc=x_raw-fed_x; yc=y_raw-fed_y
    speed=pd.to_numeric(df['Speed_cm'],errors='coerce').values if 'Speed_cm' in df.columns else np.zeros(len(df))
    vd=valid_xy & ~np.isnan(dist) & (dist>=0) & (speed<SPEED_MAX)
    if vd.sum()<MIN_REG_PTS: vd=valid_xy & ~np.isnan(dist) & (dist>=0)
    if vd.sum()<10: rec['Status']='REGRESSION_FAILED'; return df, rec
    d_px=np.sqrt(xc[vd]**2+yc[vd]**2); d_cm=dist[vd]
    denom=np.sum(d_px**2); scale=float(np.sum(d_px*d_cm)/denom) if denom>0 else np.nan
    ss_res=np.sum((d_cm-scale*d_px)**2); ss_tot=np.sum((d_cm-d_cm.mean())**2)
    r2=float(1-ss_res/ss_tot) if ss_tot>0 else np.nan
    rec['scale_cm_per_px']=round(scale,5); rec['R2']=round(r2,4)
    x_cm=xc*scale; y_cm=yc*scale
    vcm=valid_xy & np.isfinite(x_cm) & np.isfinite(y_cm)
    if vcm.sum()<10: rec['Status']='PCA_FAILED'; return df, rec
    pca=PCA(n_components=2); pca.fit(np.column_stack([x_cm[vcm],y_cm[vcm]]))
    rot=pca.components_[[1,0],:]; xyr=np.column_stack([x_cm,y_cm])@rot.T
    x_norm,y_norm=xyr[:,0],xyr[:,1]
    if np.nanmedian(y_norm[valid_xy])<0: y_norm=-y_norm
    dist_re=np.sqrt(x_norm**2+y_norm**2)
    mae=float(np.nanmean(np.abs(dist_re[vd]-dist[vd]))); rec['MAE_cm']=round(mae,3)
    status='OK'
    if r2<R2_FLAG: status=f'LOW_R2({r2:.3f})'
    elif mae>MAE_FLAG: status=f'HIGH_MAE({mae:.2f}cm)'
    rec['Status']=status
    if ANCHOR_COL not in df.columns: df[ANCHOR_COL]=df['DistFromFED']
    df['Xpos_cm']=np.where(valid_xy,x_norm,np.nan)
    df['Ypos_cm']=np.where(valid_xy,y_norm,np.nan)
    if OVERWRITE_DISTFROMFED: df['DistFromFED']=np.sqrt(df['Xpos_cm']**2+df['Ypos_cm']**2)
    return df, rec
print("Stage E function defined.")


Stage E function defined.


In [ ]:
# @title Per-animal QC figure
def qc_figure(aid, df, info, rec, out_png):
    grp=rec.get('Group','?'); sess=rec.get('Session','')
    fig,ax=plt.subplots(2,3,figsize=(16,7))
    fig.suptitle(f'{aid} s{sess} [{grp}]  norm:{rec.get("Status","-")}  R2={rec.get("R2","-")}  MAE={rec.get("MAE_cm","-")}cm',
                 fontsize=12, fontweight='bold')
    t=df['TimeStampAdjusted'].values
    # 1 raw + fits
    ax[0,0].plot(t, df['Fluorescence'],'g',lw=.4,alpha=.7,label='Fluor')
    ax[0,0].plot(t, df['Fluorescence']-df['Debleached Fluorescence'],'r',lw=.8,label='exp fit')
    ax[0,0].set_title('Raw fluorescence + debleach fit',fontsize=9); ax[0,0].legend(fontsize=7)
    # 2 z-score trace + events
    ax[0,1].plot(t, df['Corrected_Fluorescence_Z_25'],color='dimgray',lw=.4)
    for c,col in [('ToneStart','#E05C5C'),('PelletGrab','purple'),
                  ('Completed_Tone','#3C7BE0')]:
        if c in df.columns:
            for tt in t[df[c].values==1]: ax[0,1].axvline(tt,color=col,lw=.5,alpha=.5)
    ax[0,1].set_title('Z-score (25%ile) + Bonsai events',fontsize=9)
    # 3 motion correction scatter
    ax[0,2].scatter(df['Debleached UV'][::20], df['Debleached Fluorescence'][::20],s=1,alpha=.2,color='gray')
    ax[0,2].set_title('Motion fit (UV vs Fluor debleached)',fontsize=9)
    # 4 alignment drift
    if info is not None:
        mb, mr = info['mb'], info['mr']
        resid = mr - (info['slope']*mb + info['offset'])
        ax[1,0].plot(range(len(mb)), resid, 'o-', ms=3)
        ax[1,0].axhline(0,color='k',lw=.6,ls='--')
        ax[1,0].set_title(f"Align resid (n={len(mb)})  slope={info['slope']:.5f}  ({info['drift_s_per_hr']:.2f}s/hr)",fontsize=9)
        ax[1,0].set_xlabel('matched event #'); ax[1,0].set_ylabel('RWD - pred (s)')
    else:
        ax[1,0].text(.5,.5,'no Bonsai',ha='center',transform=ax[1,0].transAxes)
    # 5 raw trajectory + FED
    if 'Xpos' in df.columns:
        ax[1,1].plot(df['Xpos'],df['Ypos'],'k-',lw=.2,alpha=.4)
        if not np.isnan(rec.get('FED_x_px',np.nan)):
            ax[1,1].scatter([rec['FED_x_px']],[rec['FED_y_px']],color='red',s=50,zorder=5,label='FED est')
        ax[1,1].set_title('Raw trajectory (px) + FED',fontsize=9); ax[1,1].set_aspect('equal'); ax[1,1].legend(fontsize=7)
    # 6 normalized cm
    if 'Xpos_cm' in df.columns:
        ax[1,2].plot(df['Xpos_cm'],df['Ypos_cm'],'k-',lw=.2,alpha=.4)
        ax[1,2].scatter([0],[0],color='red',s=50,zorder=5,label='FED (0,0)')
        ax[1,2].set_title(f"Normalized cm  scale={rec.get('scale_cm_per_px','-')}",fontsize=9)
        ax[1,2].set_aspect('equal'); ax[1,2].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(out_png, dpi=110, bbox_inches='tight'); plt.close()
print("QC figure function defined.")


QC figure function defined.


In [ ]:
# @title Stage F — batch every session (skip-existing) + QC + verify + save

# Session universe = the RWD_Fluorescence files (every session has photometry).
# Keyed by session, NOT animal: each animal has two sessions and an animal-level
# key would silently keep only one of them.
rwd_files = gh_list(API_RWD_FLUOR)
rwd_index = {session_key(f['name']): f for f in rwd_files if session_key(f['name'])}
print(f"{len(rwd_index)} RWD sessions | {len(bonsai_by_session)} with Bonsai | "
      f"{len(rwd_index) - len(set(rwd_index) & set(bonsai_by_session))} photometry-only")

# Keyed by animal + session number, NOT date -- see session_key().
processed_on_github = set()
if SKIP_EXISTING:
    processed_on_github = {session_key(f['name']) for f in gh_list(API_MERGED)
                           if session_key(f['name'])}

# Committed merged data ('Processed Data'), for verification.
merged_index = {}
if VERIFY_VS_EXISTING:
    merged_index = {session_key(f['name']): f for f in gh_list(API_MERGED_REF)
                    if session_key(f['name'])}

def out_name_for(k):
    base = rwd_index[k]['name'].replace('.csv','')
    return f"{base}_merged.csv" if k in bonsai_by_session else f"{base}_merged_NOBONSAIDATA.csv"

qc_rows, verify_rows, skipped, date_mismatch = [], [], [], []
_order = sorted(rwd_index, key=lambda k: (k[0], get_animal_num(k), k))
for k in tqdm(_order, desc="Stage F: sessions"):
    meta = parse_name(rwd_index[k]['name']) or {}
    aid  = meta.get('animal', k); grp = meta.get('group','?'); sess = meta.get('session','')
    out_path = f"{OUT_DIR}/{out_name_for(k)}"
    if SKIP_EXISTING and k in processed_on_github:
        skipped.append(k); continue
    try:
        df = process_photometry(load_rwd_raw(rwd_index[k]['download_url']))
        df = add_rwd_events(df)
        ev_src = df.attrs.get('events_source','?')
        info = None
        rec = {'Animal':aid,'Group':grp,'Session':sess,'Status':'NO_BONSAI',
               'R2':np.nan,'MAE_cm':np.nan,'n_grab':0,'scale_cm_per_px':np.nan,
               'FED_x_px':np.nan,'FED_y_px':np.nan}
        if k in bonsai_by_session:
            # Flag sessions whose RWD and Bonsai filenames disagree on the date
            bmeta = parse_name(bonsai_names[k]) or {}
            if bmeta.get('date') and meta.get('date') and bmeta['date'] != meta['date']:
                date_mismatch.append(f"{k}: RWD {meta['date']} vs Bonsai {bmeta['date']}")
            df, info = align_and_merge(df, bonsai_by_session[k])
            df, rec  = normalize_positions(df, aid, grp, sess)
        rec['events'] = ev_src
        df.to_csv(out_path, index=False)
        qc_figure(aid, df, info, rec, f"{QC_DIR}/qc_{grp}_{aid}_s{sess}.png")
        row = dict(rec); row['align_r']=None if info is None else round(info['r'],4)
        row['drift_s_per_hr']=None if info is None else round(info['drift_s_per_hr'],3)
        qc_rows.append(row)

        if VERIFY_VS_EXISTING and k in merged_index:
            ref = gh_read(merged_index[k]['download_url'])
            v={'Session':k}
            if 'Corrected_Fluorescence_Z_25' in ref.columns:
                a=df['Corrected_Fluorescence_Z_25'].values; b=ref['Corrected_Fluorescence_Z_25'].values
                n=min(len(a),len(b)); v['z_corr']=round(float(np.corrcoef(a[:n],b[:n])[0,1]),5)
            for c in ['ToneStart','PelletGrab','ToneStart_Bonsai','PelletGrab_Bonsai',
                      'Completed_Tone','Completed_PelletGrab']:
                if c in df.columns and c in ref.columns:
                    v[c]=f"{int(df[c].sum())}/{int(ref[c].sum())}"
            # DistFromFED is DELIBERATELY different now (FED located from PelletGrab
            # instead of assumed at x=9cm), so compare the PRESERVED original instead
            # of treating the change as a regression.
            if 'DistFromFED' in ref.columns and 'DistFromFED_orig' in df.columns:
                a=df['DistFromFED_orig'].values; b=ref['DistFromFED'].values
                n=min(len(a),len(b)); m=np.isfinite(a[:n])&np.isfinite(b[:n])
                if m.sum()>10:
                    v['dist_orig_vs_ref']=round(float(np.corrcoef(a[:n][m],b[:n][m])[0,1]),5)
            verify_rows.append(v)
    except Exception as e:
        print(f"  x {k}: {e}")

print(f"\nProcessed {len(qc_rows)}  |  skipped (already on GitHub): {len(skipped)} {skipped}")
if skipped and not qc_rows:
    print("Nothing to do -- every session is already committed to the repo.\n"
          "Set SKIP_EXISTING=False in the options cell to rebuild them all from raw.")
if date_mismatch:
    print("\n!! RWD/Bonsai filename DATE MISMATCH (paired on animal+session anyway):")
    for _d in date_mismatch: print("   ", _d)
if qc_rows:
    print("\n-- Normalization / alignment QC --")
    print(pd.DataFrame(qc_rows)[['Animal','Session','Group','Status','events','R2','MAE_cm',
                                 'n_grab','align_r','drift_s_per_hr']].to_string(index=False))
    pd.DataFrame(qc_rows).to_csv(f"{OUT_DIR}/preprocessing_qc.csv", index=False)
if verify_rows:
    print("\n-- Verify vs committed 'Processed Data' (z corr; event counts mine/ref) --")
    print("   DistFromFED itself is EXPECTED to differ: the FED is now located from")
    print("   PelletGrab rather than assumed at x = 9 cm. dist_orig_vs_ref compares the")
    print("   preserved DistFromFED_orig against the committed column and should be ~1.0.")
    print(pd.DataFrame(verify_rows).to_string(index=False))
print(f"\nWrote merged CSVs + QC to {OUT_DIR}")


48 RWD sessions | 44 with Bonsai | 4 photometry-only


Stage F: sessions:   0%|          | 0/48 [00:00<?, ?it/s]


Processed 48  |  skipped (already on GitHub): 0 []

-- Normalization / alignment QC --
Animal Session Group           Status  events      R2  MAE_cm  n_grab  align_r  drift_s_per_hr
    F1      01   SNI    LOW_R2(0.615) carried  0.6155   2.868      14      1.0           0.512
    F1      02   SNI               OK carried  0.9377   1.254      18      1.0           0.515
    F2      01  SHAM               OK carried  0.9744   0.739       6      1.0           0.475
    F2      02  SHAM        NO_BONSAI derived     NaN     NaN       0      NaN             NaN
    F3      01  SHAM   LOW_R2(-0.513) carried -0.5128   6.193       3      NaN           0.000
    F3      02  SHAM               OK carried  0.9911   0.509      11      1.0           0.509
   F18      01  SHAM               OK carried  0.9574   1.223       8      1.0           0.557
   F18      02  SHAM               OK carried  0.9068   1.786      15      1.0           0.554
   F19      01  SHAM               OK carried  0.9750   0

In [ ]:
# @title Download the results (zip -> your machine)
# Colab-local output does not survive the runtime, so pull it down before closing.
# Commit the merged CSVs to Photometry/Pavlovian/RWD/Processed Data/
# yourself -- the notebook never writes to the repo.
import shutil
from google.colab import files

if os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR):
    _zip = shutil.make_archive('/content/PAV_Preprocessing_out', 'zip', OUT_DIR)
    print(f"Zipped -> {_zip}  ({os.path.getsize(_zip)/1e6:.0f} MB)")
    files.download(_zip)
else:
    print(f"{OUT_DIR} is empty (everything was skipped) -- nothing to download.")
